<a href="https://colab.research.google.com/github/jtGHub-svg/Fase5_Datathon/blob/main/Fase_5_Datathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Bibliotecas
import os, re, json, warnings
import matplotlib.pyplot as plt
import seaborn as sns
import pyspark
import pandas as pd
import re
from datetime import datetime
import unicodedata
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, lit, isnan, sum, round, avg
from pyspark.sql import functions as F
from pyspark.sql import Window

In [2]:
# Criar sessão Spark
spark = SparkSession.builder.appName("Desenvolvimento_Educacional").getOrCreate()

In [3]:
# Variáveis

#Caminho do arquivo excel
excel_file = "/content/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"


In [4]:
# Transforma as abas do excel em arquivos csv

# Ler todas as abas
sheets = pd.read_excel(excel_file, sheet_name=None)

# Salvar cada aba como CSV
for sheet_name, df in sheets.items():
    # Cria um nome de arquivo CSV baseado no nome da aba
    csv_file_name = f"{sheet_name}.csv"

    # Salva o DataFrame como CSV
    # index=False evita salvar a coluna de índices do Pandas
    df.to_csv(csv_file_name, index=False, encoding='utf-8-sig')
    print(f"Aba '{sheet_name}' salva como: {csv_file_name}")

Aba 'PEDE2022' salva como: PEDE2022.csv
Aba 'PEDE2023' salva como: PEDE2023.csv
Aba 'PEDE2024' salva como: PEDE2024.csv


In [5]:
# Ler os arquivos csv gerado
df_PEDE2022 = spark.read.csv("PEDE2022.csv", sep=",", header=True, inferSchema=True)
df_PEDE2023 = spark.read.csv("PEDE2023.csv", sep=",", header=True, inferSchema=True)
df_PEDE2024 = spark.read.csv("PEDE2024.csv", sep=",", header=True, inferSchema=True)

In [6]:
# Linhas e colunas PEDE2022
print('\nLinhas e colunas PEDE2022')
print('Linhas: ', df_PEDE2022.count())
print('Colunas: ', len(df_PEDE2022.columns))

# Linhas e colunas PEDE2023
print('\nLinhas e colunas PEDE2023')
print('Linhas: ', df_PEDE2023.count())
print('Colunas: ', len(df_PEDE2023.columns))

# Linhas e colunas PEDE2024
print('\nLinhas e colunas PEDE2024')
print('Linhas: ', df_PEDE2024.count())
print('Colunas: ', len(df_PEDE2024.columns))


Linhas e colunas PEDE2022
Linhas:  860
Colunas:  42

Linhas e colunas PEDE2023
Linhas:  1014
Colunas:  48

Linhas e colunas PEDE2024
Linhas:  1156
Colunas:  50


In [7]:
# Verificação de Nomes, Ordem e Tipos de Dados

# Mostra esquema
df_PEDE2022.printSchema()

# Mostra primeiras linhas
df_PEDE2022.show(5)

root
 |-- RA: string (nullable = true)
 |-- Fase: integer (nullable = true)
 |-- Turma: string (nullable = true)
 |-- Nome: string (nullable = true)
 |-- Ano nasc: integer (nullable = true)
 |-- Idade 22: integer (nullable = true)
 |-- Gênero: string (nullable = true)
 |-- Ano ingresso: integer (nullable = true)
 |-- Instituição de ensino: string (nullable = true)
 |-- Pedra 20: string (nullable = true)
 |-- Pedra 21: string (nullable = true)
 |-- Pedra 22: string (nullable = true)
 |-- INDE 22: double (nullable = true)
 |-- Cg: integer (nullable = true)
 |-- Cf: integer (nullable = true)
 |-- Ct: integer (nullable = true)
 |-- Nº Av: integer (nullable = true)
 |-- Avaliador1: string (nullable = true)
 |-- Rec Av1: string (nullable = true)
 |-- Avaliador2: string (nullable = true)
 |-- Rec Av2: string (nullable = true)
 |-- Avaliador3: string (nullable = true)
 |-- Rec Av3: string (nullable = true)
 |-- Avaliador4: string (nullable = true)
 |-- Rec Av4: string (nullable = true)
 |-- IA

In [8]:
# Mostra esquema
df_PEDE2023.printSchema()

# Mostra primeiras linhas
df_PEDE2023.show(5)

root
 |-- RA: string (nullable = true)
 |-- Fase: string (nullable = true)
 |-- INDE 2023: double (nullable = true)
 |-- Pedra 2023: string (nullable = true)
 |-- Turma: string (nullable = true)
 |-- Nome Anonimizado: string (nullable = true)
 |-- Data de Nasc: string (nullable = true)
 |-- Idade: string (nullable = true)
 |-- Gênero: string (nullable = true)
 |-- Ano ingresso: integer (nullable = true)
 |-- Instituição de ensino: string (nullable = true)
 |-- Pedra 20: string (nullable = true)
 |-- Pedra 21: string (nullable = true)
 |-- Pedra 22: string (nullable = true)
 |-- Pedra 23: string (nullable = true)
 |-- INDE 22: double (nullable = true)
 |-- INDE 23: string (nullable = true)
 |-- Cg: string (nullable = true)
 |-- Cf: string (nullable = true)
 |-- Ct: string (nullable = true)
 |-- Nº Av: double (nullable = true)
 |-- Avaliador1: string (nullable = true)
 |-- Rec Av1: string (nullable = true)
 |-- Avaliador2: string (nullable = true)
 |-- Rec Av2: string (nullable = true)
 

In [9]:
# Mostra esquema
df_PEDE2024.printSchema()

# Mostra primeiras linhas
df_PEDE2024.show(5)

root
 |-- RA: string (nullable = true)
 |-- Fase: string (nullable = true)
 |-- INDE 2024: string (nullable = true)
 |-- Pedra 2024: string (nullable = true)
 |-- Turma: string (nullable = true)
 |-- Nome Anonimizado: string (nullable = true)
 |-- Data de Nasc: timestamp (nullable = true)
 |-- Idade: integer (nullable = true)
 |-- Gênero: string (nullable = true)
 |-- Ano ingresso: integer (nullable = true)
 |-- Instituição de ensino: string (nullable = true)
 |-- Pedra 20: string (nullable = true)
 |-- Pedra 21: string (nullable = true)
 |-- Pedra 22: string (nullable = true)
 |-- Pedra 23: string (nullable = true)
 |-- INDE 22: double (nullable = true)
 |-- INDE 23: double (nullable = true)
 |-- Cg: string (nullable = true)
 |-- Cf: string (nullable = true)
 |-- Ct: string (nullable = true)
 |-- Nº Av: integer (nullable = true)
 |-- Avaliador1: string (nullable = true)
 |-- Rec Av1: string (nullable = true)
 |-- Avaliador2: string (nullable = true)
 |-- Rec Av2: string (nullable = tr

In [10]:
# -*- coding: utf-8 -*-
"""
Limpeza e consolidação dos arquivos PEDE 2022–2024
Regras aplicadas:
 1) Inclui Ano_Dados
 2) Remove Pedra 20 e Pedra 21 (todos anos)
 3) Remove Pedra 22, Pedra 23, INDE 22, INDE 23 (apenas 2023 e 2024)
 4) Remove Destaque IPV.1 (2023)
 5) Remove Avaliador5, Avaliador6, Escola, Ativo/ Inativo e Ativo/ Inativo.1 (2024)
 6) Padroniza nomes de colunas (substitui espaços/pontos/“º” por _ e remove sufixos de anos)
 7) Em 2023 recalcula Idade = 2023 - ano(Data de Nasc)
 8) Renomeia "Nome Anonimizado" → "Nome"
 9) Padroniza nomes de campos para todos os anos
10) Converte strings numéricas para numérico quando fizer sentido
11) Padroniza Fase (ALFA→0; “Fase n (…)”→n; “nA..nX”→n)
"""

# ---------- utilitários ----------
def carregar_csv(nome_arquivo: str) -> pd.DataFrame:
    try:
        return pd.read_csv(nome_arquivo, dtype=str, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(nome_arquivo, dtype=str, encoding="latin-1")

def limpar_nome_coluna(col: str) -> str:
    s = col.replace("º", "").replace(".", "").replace(" ", "_")
    s = s.replace("/", "_").replace("-", "_")
    s = re.sub(r"_+", "_", s)
    s = re.sub(r"_(19|20)\d{2}$", "", s)  # remove sufixo _YYYY
    s = re.sub(r"_\d{2}$", "", s)         # remove sufixo _YY
    return s.strip("_")

def padronizar_fase(valor):
    if pd.isna(valor):
        return None
    v = str(valor).strip().upper()
    if v == "ALFA":
        return 0
    m = re.search(r"FASE\s*(\d+)", v)
    if m:
        return int(m.group(1))
    m2 = re.match(r"^(\d+)[A-Z]+$", v)
    if m2:
        return int(m2.group(1))
    try:
        return int(float(v))
    except ValueError:
        return None

def forcar_numericos(df: pd.DataFrame) -> pd.DataFrame:
    candidatos_forcados = {
        "IAA","IEG","IPS","IPP","IDA","IAN","IPV","INDE","Cg","Cf","Ct",
        "Mat","Por","Ing","Defasagem","N_Av","Ano_ingresso","Ano_nasc","Idade",
        "INDE_2022","INDE_2023","INDE_2024"
    }
    novos = {}
    for col in df.columns:
        # Mantém categóricas como texto
        if col in {
            "RA","Turma","Nome","Nome_Anonimizado","Genero","Instituicao_de_ensino",
            "Pedra","Pedra_20","Pedra_21","Pedra_22","Pedra_23","Avaliador1","Avaliador2",
            "Avaliador3","Avaliador4","Avaliador5","Avaliador6","Rec_Av1","Rec_Av2",
            "Rec_Av3","Rec_Av4","Rec_Psicologia","Indicado","Fase_Ideal","Destaque_IEG",
            "Destaque_IDA","Destaque_IPV","Escola","Ativo_Inativo","Ativo_Inativo_1"
        }:
            continue
        serie = df[col]
        if serie.dtype == object:
            s = serie.str.replace(",", ".", regex=False)
            num = pd.to_numeric(s, errors="coerce")
            non_na = num.notna().sum()
            orig_non_na = serie.notna().sum()
            proporcao = (non_na / orig_non_na) if orig_non_na else 0
            if proporcao >= 0.8 or col in candidatos_forcados or re.search(r"(\d)", col):
                novos[col] = num
    if novos:
        df = df.assign(**novos)
    return df

# Padronizar campo datas de nascimento
def padronizar_data_nascimento(series: pd.Series):
    def parse_one(x):
        if pd.isna(x): return None
        s = str(x).strip()
        if not s: return None
        s = re.sub(r"\s+\d{2}:\d{2}:\d{2}$", "", s)
        for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%Y/%m/%d"):
            try: return datetime.strptime(s, fmt).date()
            except ValueError: pass
        m = re.match(r"^(\d{4})", s)
        if m: return datetime(int(m.group(1)), 1, 1).date()
        return None
    return series.apply(parse_one)

# processar arquivos csv gerados a partir do arquivo excel
def processar(arquivo: str, ano_dados: int) -> pd.DataFrame:
    df = carregar_csv(arquivo)
    # 1) Ano_Dados
    df["Ano_Dados"] = str(ano_dados)

    # Remove as colunas Pedra 20 e 21
    for c in ["Pedra 20","Pedra 21"]:
        if c in df.columns:
            df = df.drop(columns=[c])

    # Remove as colunas Pedra 22, Pedra 23, INDE 22 e INDE 23 (2023/2024)
    if ano_dados in (2023, 2024):
        for c in ["Pedra 22","Pedra 23","INDE 22","INDE 23"]:
            if c in df.columns:
                df = df.drop(columns=[c])

    # Remove a coluna Destaque IPV.1 (2023)
    if ano_dados == 2023 and "Destaque IPV.1" in df.columns:
        df = df.drop(columns=["Destaque IPV.1"])

    # Remove colunas Avaliador5, Avaliador6, Escola, Ativo/ Inativo e Ativo/ Inativo.1 (2024)
    if ano_dados == 2024:
        for c in ["Avaliador5","Avaliador6","Escola","Ativo/ Inativo","Ativo/ Inativo.1"]:
            if c in df.columns:
                df = df.drop(columns=[c])

    # Altera o campo Nome Anonimizado para Nome e ajustes de acentuação
    renomes = {}
    if "Nome Anonimizado" in df.columns:
        renomes["Nome Anonimizado"] = "Nome"
    if "Gênero" in df.columns:
        renomes["Gênero"] = "Genero"
    if "Instituição de ensino" in df.columns:
        renomes["Instituição de ensino"] = "Instituicao de ensino"
    if "Data de Nasc" in df.columns:
        renomes["Data de Nasc"] = "Data de Nasc"
    if renomes:
        df = df.rename(columns=renomes)

    # Padroniza cabeçalho
    df = df.rename(columns={c: limpar_nome_coluna(c) for c in df.columns})

    # Calcula Idade 2023 a partir de Data_de_Nasc e
    # calcula o ano de nascimento dos anos 2022 e 2024 a partir data
    # de nascimento e deleta campo data de nascimento
    ano_nasc_series = pd.Series([None]*len(df))
    if "Data_de_Nasc" in df.columns:
        parsed = padronizar_data_nascimento(df["Data_de_Nasc"])
        ano_nasc_series = parsed.apply(lambda d: d.year if d else None)
        if ano_dados == 2023:
            df["Idade"] = parsed.apply(lambda d: 2023 - d.year if d else None)
        df = df.drop(columns=["Data_de_Nasc"])
    else:
        if "Ano_nasc" in df.columns:
            ano_nasc_series = pd.to_numeric(df["Ano_nasc"], errors="coerce")
        elif "Idade" in df.columns and df["Idade"].notna().any():
            idade_num = pd.to_numeric(df["Idade"], errors="coerce")
            ano_ref = int(df["Ano_Dados"].iloc[0])
            ano_nasc_series = ano_ref - idade_num
    df["Ano_nasc"] = ano_nasc_series

    # 11) Fase
    if "Fase" in df.columns:
        df["Fase"] = df["Fase"].apply(padronizar_fase)

    # 10) Forçar numéricos
    df = forcar_numericos(df)

    return df

# ---------- executa para 2022, 2023, 2024 ----------
arquivos = [("PEDE2022.csv", 2022), ("PEDE2023.csv", 2023), ("PEDE2024.csv", 2024)]
processados = {}
for nome, ano in arquivos:
    processados[ano] = processar(nome, ano)

# Harmonização final dos nomes (sinônimos)
CANONICOS = {
    "Matem": "Mat",
    "Portug": "Por",
    "Ingles": "Ing",     # caso sem acento
    "Inglês": "Ing",     # caso com acento
    "Fase_ideal": "Fase_Ideal",
    "Defas": "Defasagem",
}
for ano, df in processados.items():
    processados[ano] = df.rename(columns={c: CANONICOS.get(c, c) for c in df.columns})

# Ordem final das colunas (idêntica nos três arquivos)
todas = set()
for df in processados.values():
    todas.update(df.columns)
base_ordem = [
    "RA","Nome","Fase","Turma","Ano_Dados","Ano_nasc","Idade","Genero",
    "Ano_ingresso","Instituicao_de_ensino","Pedra","INDE","Cg","Cf","Ct","N_Av",
    "Avaliador1","Rec_Av1","Avaliador2","Rec_Av2","Avaliador3","Rec_Av3","Avaliador4","Rec_Av4",
    "IAA","IEG","IPS","IPP","Rec_Psicologia","IDA","Mat","Por","Ing","Indicado","Atingiu_PV",
    "IPV","IAN","Fase_Ideal","Defasagem","Destaque_IEG","Destaque_IDA","Destaque_IPV"
]
ordem = [c for c in base_ordem if c in todas] + sorted([c for c in todas if c not in base_ordem])

# Salva 3 arquivos limpos
for ano, df in processados.items():
    df = df.reindex(columns=ordem)
    df.to_csv(f"PEDE{ano}_limpo.csv", index=False, encoding="utf-8-sig")

# Consolida (concatena) os três anos
dfs = [processados[2022].reindex(columns=ordem),
       processados[2023].reindex(columns=ordem),
       processados[2024].reindex(columns=ordem)]
consolidado = pd.concat(dfs, ignore_index=True).sort_values(["Ano_Dados","RA"], ignore_index=True)

# Exporta consolidado
consolidado.to_csv("PEDE_consolidado_2022_2024.csv", index=False, encoding="utf-8-sig")
consolidado.to_parquet("PEDE_consolidado_2022_2024.parquet", index=False)

print("Concluído!")

Concluído!


In [21]:
# Ler o arquivo consolidado
df_consolidado = spark.read.csv("PEDE_consolidado_2022_2024.csv", sep=",", header=True, inferSchema=True)

# Mostra esquema
df_consolidado.printSchema()

# Mostra primeiras linhas
df_consolidado.show(5)
df_consolidado.orderBy("Ano_Dados", ascending=False).show(5)

root
 |-- RA: string (nullable = true)
 |-- Nome: string (nullable = true)
 |-- Fase: integer (nullable = true)
 |-- Turma: string (nullable = true)
 |-- Ano_Dados: integer (nullable = true)
 |-- Ano_nasc: integer (nullable = true)
 |-- Idade: integer (nullable = true)
 |-- Genero: string (nullable = true)
 |-- Ano_ingresso: integer (nullable = true)
 |-- Instituicao_de_ensino: string (nullable = true)
 |-- Pedra: string (nullable = true)
 |-- INDE: double (nullable = true)
 |-- Cg: double (nullable = true)
 |-- Cf: double (nullable = true)
 |-- Ct: double (nullable = true)
 |-- N_Av: double (nullable = true)
 |-- Avaliador1: string (nullable = true)
 |-- Rec_Av1: string (nullable = true)
 |-- Avaliador2: string (nullable = true)
 |-- Rec_Av2: string (nullable = true)
 |-- Avaliador3: string (nullable = true)
 |-- Rec_Av3: string (nullable = true)
 |-- Avaliador4: string (nullable = true)
 |-- Rec_Av4: string (nullable = true)
 |-- IAA: double (nullable = true)
 |-- IEG: double (nullab

In [12]:
# Linhas e colunas df_consolidado
print('\nLinhas e colunas df_consolidado')
print('Linhas: ', df_consolidado.count())
print('Colunas: ', len(df_consolidado.columns))


Linhas e colunas df_consolidado
Linhas:  3030
Colunas:  42


In [13]:
# Regras de PEDRA (faixas por INDE), segundo dicionário PEDE
# Quartzo – 2,405 a 5,506 | Ágata – 5,506 a 6,868 | Ametista – 6,868 a 8,230 | Topázio – 8,230 a 9,294
PEDRA_INTERVALOS = [
    ('Quartzo', 2.405, 5.506),
    ('Ágata', 5.506, 6.868),
    ('Ametista', 6.868, 8.230),
    ('Topázio', 8.230, 9.294),
]

In [14]:
# Verificação de valores distintos (duplicidade)
print(f'Quantidade de Linhas: {df_consolidado.count()}')
print(f'Linhas distintas: {df_consolidado.distinct().count()}')

Quantidade de Linhas: 3030
Linhas distintas: 3030


In [18]:
# Estatísticas descritivas
df_consolidado.describe().show()

+-------+------+---------+------------------+-----------------+------------------+------------------+------------------+--------+------------------+---------------------+-----+------------------+------------------+-----------------+-----------------+------------------+------------+--------------------+-----------+--------------------+-----------+--------------------+-----------+--------------------+------------------+-----------------+------------------+-----------------+--------------+------------------+------------------+------------------+-----------------+--------+----------+-----------------+------------------+--------------------+------------------+--------------------+--------------------+--------------------+
|summary|    RA|     Nome|              Fase|            Turma|         Ano_Dados|          Ano_nasc|             Idade|  Genero|      Ano_ingresso|Instituicao_de_ensino|Pedra|              INDE|                Cg|               Cf|               Ct|              N_Av|  Av

In [17]:
df_consolidado.summary().show()  # count, mean, stddev, min, 25%, 50%, 75%, max

# Seleção customizada
df_consolidado.summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

+-------+------+---------+------------------+-----------------+------------------+------------------+------------------+--------+------------------+---------------------+-----+------------------+------------------+-----------------+-----------------+------------------+------------+--------------------+-----------+--------------------+-----------+--------------------+-----------+--------------------+------------------+-----------------+------------------+-----------------+--------------+------------------+------------------+------------------+-----------------+--------+----------+-----------------+------------------+--------------------+------------------+--------------------+--------------------+--------------------+
|summary|    RA|     Nome|              Fase|            Turma|         Ano_Dados|          Ano_nasc|             Idade|  Genero|      Ano_ingresso|Instituicao_de_ensino|Pedra|              INDE|                Cg|               Cf|               Ct|              N_Av|  Av